# Silver Layer: CRM Sales Details

**Source**: `databricks_bootcamp_dwb.bronze.crm_sales_details`  
**Target**: `databricks_bootcamp_dwb.silver.crm_sales`  
**Architecture**: Medallion (Bronze → Silver)

## Data Quality Analysis Summary

### Identified Issues:
1. **No Duplicates**: Each order-product combination is unique (60,398 unique combinations)
2. **Date Format**: Dates stored as integers in YYYYMMDD format, need conversion
3. **Zero Dates**: Some date columns contain 0 values (invalid dates)
4. **Column Names**: Abbreviated column names need to be expanded for readability
5. **Data Types**: Monetary values stored as integers, need DECIMAL type

## Transformation Plan:
1. Read bronze data into DataFrame
2. Analyze null values across all columns
3. Convert date columns from integer to DATE (handle 0s as NULL)
4. Cast numeric columns to appropriate types (DECIMAL for currency)
5. Rename columns to readable names
6. Validate final data quality
7. Write to silver table

In [0]:
-- Sample bronze data to understand structure
SELECT * 
FROM databricks_bootcamp_dwb.bronze.crm_sales_details
LIMIT 50

In [0]:
-- Check for duplicates and data quality issues
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT sls_ord_num) as unique_orders,
  COUNT(DISTINCT CONCAT(sls_ord_num, '-', sls_prd_key)) as unique_order_product_combinations,
  COUNT(*) - COUNT(DISTINCT CONCAT(sls_ord_num, '-', sls_prd_key)) as potential_duplicates,
  MIN(sls_order_dt) as earliest_order_date,
  MAX(sls_order_dt) as latest_order_date
FROM databricks_bootcamp_dwb.bronze.crm_sales_details

## Data Quality Analysis
Detailed analysis of data quality issues in the bronze table.

In [0]:
-- Check for null values in each column
SELECT 
  COUNT(*) - COUNT(sls_ord_num) as null_order_num,
  COUNT(*) - COUNT(sls_prd_key) as null_product_key,
  COUNT(*) - COUNT(sls_cust_id) as null_customer_id,
  COUNT(*) - COUNT(sls_order_dt) as null_order_date,
  COUNT(*) - COUNT(sls_ship_dt) as null_ship_date,
  COUNT(*) - COUNT(sls_due_dt) as null_due_date,
  COUNT(*) - COUNT(sls_sales) as null_sales,
  COUNT(*) - COUNT(sls_quantity) as null_quantity,
  COUNT(*) - COUNT(sls_price) as null_price
FROM databricks_bootcamp_dwb.bronze.crm_sales_details

In [0]:
-- Preview records with null prices and/or null sales
SELECT *
FROM databricks_bootcamp_dwb.bronze.crm_sales_details
WHERE sls_price IS NULL OR sls_sales IS NULL
LIMIT 20

In [0]:
SELECT * 
FROM databricks_bootcamp_dwb.bronze.crm_sales_details
WHERE sls_quantity > 1

In [0]:
-- Check for zero dates (these are invalid and should be converted to NULL)
SELECT 
  SUM(CASE WHEN sls_order_dt = 0 THEN 1 ELSE 0 END) as zero_order_dates,
  SUM(CASE WHEN sls_ship_dt = 0 THEN 1 ELSE 0 END) as zero_ship_dates,
  SUM(CASE WHEN sls_due_dt = 0 THEN 1 ELSE 0 END) as zero_due_dates,
  COUNT(*) as total_records
FROM databricks_bootcamp_dwb.bronze.crm_sales_details

In [0]:
-- Show sample records with zero dates
SELECT *
FROM databricks_bootcamp_dwb.bronze.crm_sales_details
WHERE sls_order_dt = 0 OR sls_ship_dt = 0 OR sls_due_dt = 0
LIMIT 20

---
## Section 1: Read Bronze Data
Load the bronze table into a DataFrame for transformation.

In [0]:
%python
# Read bronze table into DataFrame
df = spark.table("databricks_bootcamp_dwb.bronze.crm_sales_details")

print(f"Total records: {df.count()}")
print("\nSchema:")
df.printSchema()
display(df)

---
## Section 2: Analyze Null Values
Check for null values in the DataFrame.

In [0]:
%python
from pyspark.sql.functions import sum, col

# Count nulls in each column
null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]

print("Null counts per column:")
for col_name in df.columns:
    print(f"  {col_name}: {null_counts[col_name]}")

---
## Section 3: Convert Date Columns
Convert integer dates (YYYYMMDD) to proper DATE type. Treat 0 dates as invalid and convert to NULL.

In [0]:
%python
from pyspark.sql.functions import when, col, expr

# Convert integer dates to string then to date
# Use try_to_date to handle invalid dates gracefully (converts to NULL)
# Also explicitly handle 0 values
df = df.withColumn(
    "sls_order_dt",
    when(col("sls_order_dt") == 0, None)
    .otherwise(expr("try_to_date(cast(sls_order_dt as string), 'yyyyMMdd')"))
).withColumn(
    "sls_ship_dt",
    when(col("sls_ship_dt") == 0, None)
    .otherwise(expr("try_to_date(cast(sls_ship_dt as string), 'yyyyMMdd')"))
).withColumn(
    "sls_due_dt",
    when(col("sls_due_dt") == 0, None)
    .otherwise(expr("try_to_date(cast(sls_due_dt as string), 'yyyyMMdd')"))
)

print("Date columns converted to DATE type (zeros converted to NULL)")
df.printSchema()
display(df)

In [0]:
%python
# Verify date conversion - check null counts after conversion
date_summary = df.select(
    sum(col("sls_order_dt").isNull().cast("int")).alias("null_order_dates"),
    sum(col("sls_ship_dt").isNull().cast("int")).alias("null_ship_dates"),
    sum(col("sls_due_dt").isNull().cast("int")).alias("null_due_dates")
)

print("Date conversion summary:")
display(date_summary)

---
## Section 4: Cast Numeric Columns
Convert monetary columns to DECIMAL type for proper precision.

In [0]:
%python
from pyspark.sql.types import DecimalType

# Cast monetary columns to DECIMAL(10, 2)
df = df.withColumn("sls_sales", col("sls_sales").cast(DecimalType(10, 2))) \
       .withColumn("sls_price", col("sls_price").cast(DecimalType(10, 2)))

print("Numeric columns cast to proper types")
df.printSchema()
display(df)

---
## Section 5: Rename Columns
Rename abbreviated column names to readable business terms.

In [0]:
%python
# Rename columns to more readable names
df_clean = df \
    .withColumnRenamed("sls_ord_num", "order_number") \
    .withColumnRenamed("sls_prd_key", "product_key") \
    .withColumnRenamed("sls_cust_id", "customer_id") \
    .withColumnRenamed("sls_order_dt", "order_date") \
    .withColumnRenamed("sls_ship_dt", "ship_date") \
    .withColumnRenamed("sls_due_dt", "due_date") \
    .withColumnRenamed("sls_sales", "sales_amount") \
    .withColumnRenamed("sls_quantity", "quantity") \
    .withColumnRenamed("sls_price", "unit_price")

print("Columns renamed")
df_clean.printSchema()
display(df_clean)

---
## Sanity Checks
Validate the final DataFrame before writing to silver layer.

In [0]:
%python
from pyspark.sql.functions import count

print("=== Final Data Quality Checks ===")
print(f"\nTotal records: {df_clean.count()}")
print(f"Unique order_numbers: {df_clean.select('order_number').distinct().count()}")
print(f"Unique product_keys: {df_clean.select('product_key').distinct().count()}")
print(f"Unique customers: {df_clean.select('customer_id').distinct().count()}")

# Check null distribution after transformations
null_summary = df_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_clean.columns])
print("\nNull counts after transformation:")
display(null_summary)

In [0]:
%python
# Show sample of cleaned data
print("Sample of final cleaned data:")
display(df_clean.limit(20))

---
## Write to Silver Layer
Write the cleaned and transformed data to the silver table.

In [0]:
%python
# Write to silver table with schema overwrite
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_bootcamp_dwb.silver.crm_sales")

print("✅ Data successfully written to databricks_bootcamp_dwb.silver.crm_sales")

In [0]:
-- Verify the silver table was created successfully
SELECT 
  COUNT(*) as total_records,
  MIN(order_date) as earliest_order,
  MAX(order_date) as latest_order,
  SUM(sales_amount) as total_sales,
  SUM(quantity) as total_quantity,
  COUNT(DISTINCT customer_id) as unique_customers,
  COUNT(DISTINCT product_key) as unique_products
FROM databricks_bootcamp_dwb.silver.crm_sales

In [0]:
-- View sample of transformed data
SELECT * 
FROM databricks_bootcamp_dwb.silver.crm_sales 
ORDER BY order_date DESC
LIMIT 20

---
---
# APPENDIX: Product Price Analysis

This section contains exploratory analysis comparing sales prices to product costs. The investigation was conducted to understand the relationship between `sls_price`, `sls_sales`, and `prd_cost` columns across the sales and product tables.

**Purpose**: Verify hypothesis about price column relationships when quantity > 1.

**Key Finding**: Both `sls_price` and `sls_sales` represent unit selling price (not total amount). The hypothesis that they should differ based on quantity was disproven.

---
## Investigation: Product Key Relationship

**Hypothesis**: The `sls_prd_key` values in sales data may be abbreviated versions of `prd_key` values in the product info table.

**Goal**: Determine if there's a pattern that explains why direct joins between the tables failed (0 matches found).

**Analysis Plan**:
1. Sample product keys from both tables
2. Look for substring/pattern relationships
3. Check if sls_prd_key appears as a substring within prd_key

In [0]:
%python
from pyspark.sql.functions import col, length, substring

# Get sample keys from both tables for comparison
print("=== Product Keys from crm_sales_details ===")
sales_keys = spark.table("databricks_bootcamp_dwb.bronze.crm_sales_details") \
    .select("sls_prd_key").distinct() \
    .withColumn("key_length", length(col("sls_prd_key"))) \
    .orderBy("sls_prd_key")

print(f"Total distinct sales product keys: {sales_keys.count()}")
print("\nSample sales keys:")
display(sales_keys.limit(20))

print("\n=== Product Keys from crm_prd_info ===")
product_keys = spark.table("databricks_bootcamp_dwb.bronze.crm_prd_info") \
    .select("prd_key").distinct() \
    .withColumn("key_length", length(col("prd_key"))) \
    .orderBy("prd_key")

print(f"Total distinct product keys: {product_keys.count()}")
print("\nSample product keys:")
display(product_keys.limit(20))

In [0]:
%python
print("=== Checking for substring relationships ===")

# Get all distinct keys
sales_keys_list = spark.table("databricks_bootcamp_dwb.bronze.crm_sales_details") \
    .select("sls_prd_key").distinct().collect()
product_keys_df = spark.table("databricks_bootcamp_dwb.bronze.crm_prd_info") \
    .select("prd_key").distinct()

print(f"\nAnalyzing {len(sales_keys_list)} sales keys against product keys...\n")

# For each sales key, check if it appears as a substring in any product key
from pyspark.sql.functions import lit, when

matches_found = 0
sample_matches = []

for row in sales_keys_list[:10]:  # Check first 10 as a sample
    sales_key = row["sls_prd_key"]
    
    # Check if this sales key is a substring of any product key
    matches = product_keys_df.filter(col("prd_key").contains(sales_key))
    match_count = matches.count()
    
    if match_count > 0:
        matches_found += 1
        sample_matches.append({
            "sales_key": sales_key,
            "matches": match_count,
            "product_keys": [m["prd_key"] for m in matches.limit(3).collect()]
        })
        
if matches_found > 0:
    print(f"✅ Found {matches_found} sales keys that appear as substrings in product keys!\n")
    print("Sample matches:")
    for match in sample_matches:
        print(f"\nSales key: {match['sales_key']}")
        print(f"  Matches {match['matches']} product key(s):")
        for pk in match['product_keys']:
            print(f"    - {pk}")
else:
    print("❌ No substring matches found in sample.")
    print("\nThe keys use different formats/naming conventions.")
    print("\nSample comparison:")
    print(f"  Sales key example: {sales_keys_list[0]['sls_prd_key']}")
    product_sample = product_keys_df.limit(1).collect()[0]['prd_key']
    print(f"  Product key example: {product_sample}")

In [0]:
%python
print("=== Analyzing key structure patterns ===")

# Analyze the structure of keys from both tables
from pyspark.sql.functions import regexp_extract, split, size

print("\n--- Sales Keys (sls_prd_key) ---")
sales_structure = spark.table("databricks_bootcamp_dwb.bronze.crm_sales_details") \
    .select("sls_prd_key") \
    .distinct() \
    .withColumn("num_segments", size(split(col("sls_prd_key"), "-"))) \
    .withColumn("key_length", length(col("sls_prd_key")))

print("Key length distribution:")
sales_structure.groupBy("key_length").count().orderBy("key_length").show()

print("Segment count distribution (separated by '-'):")
sales_structure.groupBy("num_segments").count().orderBy("num_segments").show()

print("\n--- Product Keys (prd_key) ---")
product_structure = spark.table("databricks_bootcamp_dwb.bronze.crm_prd_info") \
    .select("prd_key") \
    .distinct() \
    .withColumn("num_segments", size(split(col("prd_key"), "-"))) \
    .withColumn("key_length", length(col("prd_key")))

print("Key length distribution:")
product_structure.groupBy("key_length").count().orderBy("key_length").show()

print("Segment count distribution (separated by '-'):")
product_structure.groupBy("num_segments").count().orderBy("num_segments").show()

---
## Key Finding: Product Key Relationship Discovered

### Pattern Identified

**The sales product keys (`sls_prd_key`) ARE substrings of the product table keys (`prd_key`)!**

### Structure

* **Sales keys**: 2-3 segments, 7-10 characters
  * Examples: `BK-R93R-62`, `BK-M82S-44`, `BC-M005`

* **Product keys**: 4-5 segments, 13-16 characters  
  * Examples: `BI-RB-BK-R93R-62`, `BI-MB-BK-M82S-44`, `AC-BC-BC-M005`

### Relationship Pattern

```
Product Key Format: [CATEGORY_PREFIX]-[SALES_KEY]

Examples:
  Sales: BK-R93R-62  →  Product: BI-RB-BK-R93R-62  (prefix: BI-RB-)
  Sales: BK-M82S-44  →  Product: BI-MB-BK-M82S-44  (prefix: BI-MB-)
  Sales: BC-M005     →  Product: AC-BC-BC-M005     (prefix: AC-BC-)
```

### Join Strategy

To successfully join the tables, use one of these approaches:

1. **Substring match**: `prd_key LIKE CONCAT('%', sls_prd_key)`
2. **Extract suffix**: Strip the prefix from `prd_key` and match to `sls_prd_key`
3. **Add derived column**: Create a `prd_key_short` column in the product table by extracting the suffix

### Impact on Original Hypothesis Verification

Now we can properly join to `crm_prd_info` and compare `sls_price` and `sls_sales` against `prd_cost`.

---
## Hypothesis Retest: Using Substring Matching

**Hypothesis**: When `quantity > 1`, sls_sales and sls_price should NOT be equal. Instead:
* `sls_price = sls_sales / quantity`, OR
* `sls_sales = sls_price * quantity`

**Verification Method**: Join sales data to product data using substring matching, then compare `sls_price` and `sls_sales` against `prd_cost` to identify patterns.

In [0]:
%python
from pyspark.sql.functions import col, when

# Load both tables
sales_df = spark.table("databricks_bootcamp_dwb.bronze.crm_sales_details")
product_df = spark.table("databricks_bootcamp_dwb.bronze.crm_prd_info")

print(f"Sales records: {sales_df.count()}")
print(f"Product records: {product_df.count()}")

# Join using substring matching: prd_key contains sls_prd_key
joined_df = sales_df.alias("s").join(
    product_df.alias("p"),
    col("p.prd_key").contains(col("s.sls_prd_key")),
    "inner"
).select(
    col("s.sls_ord_num").alias("order_num"),
    col("s.sls_prd_key").alias("sales_key"),
    col("p.prd_key").alias("product_key"),
    col("p.prd_cost").alias("product_cost"),
    col("s.sls_price").alias("sales_price"),
    col("s.sls_sales").alias("sales_amount"),
    col("s.sls_quantity").alias("quantity")
)

print(f"\nJoined records: {joined_df.count()}")
print("\nSample joined data:")
display(joined_df.limit(20))

In [0]:
%python
print("=== Analysis for quantity = 1 ===")
print("These should show the relationship between prd_cost, sls_price, and sls_sales\n")

# Filter to quantity = 1
qty_1_df = joined_df.filter(col("quantity") == 1)

print(f"Records with quantity = 1: {qty_1_df.count()}\n")

# Add comparison columns
analysis_qty1 = qty_1_df.withColumn(
    "price_equals_cost",
    when(col("sales_price") == col("product_cost"), "Match").otherwise("Mismatch")
).withColumn(
    "sales_equals_cost",
    when(col("sales_amount") == col("product_cost"), "Match").otherwise("Mismatch")
).withColumn(
    "price_equals_sales",
    when(col("sales_price") == col("sales_amount"), "Match").otherwise("Mismatch")
)

print("Sample rows (quantity = 1):")
display(analysis_qty1.orderBy("order_num").limit(30))

# Summary statistics
print("\nComparison Summary for quantity = 1:")
analysis_qty1.groupBy("price_equals_cost", "sales_equals_cost", "price_equals_sales").count().show()

In [0]:
%python
print("=== Analysis for quantity > 1 ===")
print("Testing hypothesis: sls_price and sls_sales should follow a quantity relationship\n")

# Filter to quantity > 1
qty_gt1_df = joined_df.filter(col("quantity") > 1)

print(f"Records with quantity > 1: {qty_gt1_df.count()}\n")

# Add calculated columns to test hypothesis
analysis_qtymore = qty_gt1_df.withColumn(
    "price_times_qty",
    col("sales_price") * col("quantity")
).withColumn(
    "sales_div_qty",
    col("sales_amount") / col("quantity")
).withColumn(
    "hypothesis_1",
    when(col("sales_amount") == (col("sales_price") * col("quantity")), "sales = price * qty").otherwise("No match")
).withColumn(
    "hypothesis_2",
    when(col("sales_price") == (col("sales_amount") / col("quantity")), "price = sales / qty").otherwise("No match")
).withColumn(
    "price_vs_cost",
    when(col("sales_price") == col("product_cost"), "price = cost")
    .when(col("sales_amount") == col("product_cost"), "sales = cost")
    .otherwise("neither")
)

print("All records with quantity > 1:")
display(analysis_qtymore.orderBy("order_num"))

# Summary
print("\nHypothesis Test Results:")
analysis_qtymore.groupBy("hypothesis_1").count().show()
print("")
analysis_qtymore.groupBy("hypothesis_2").count().show()
print("")
analysis_qtymore.groupBy("price_vs_cost").count().show()

In [0]:
%python
print("=== Detailed Pattern Analysis ===")
print("Examining the relationship between product_cost, sales_price, and sales_amount\n")

# Look at all records with non-null values
pattern_df = joined_df.filter(
    col("product_cost").isNotNull() & 
    col("sales_price").isNotNull() & 
    col("sales_amount").isNotNull()
).withColumn(
    "cost_vs_price",
    when(col("product_cost") == col("sales_price"), "cost = price")
    .when(col("product_cost") > col("sales_price"), "cost > price")
    .when(col("product_cost") < col("sales_price"), "cost < price")
    .otherwise("unknown")
).withColumn(
    "cost_vs_sales",
    when(col("product_cost") == col("sales_amount"), "cost = sales")
    .when(col("product_cost") > col("sales_amount"), "cost > sales")
    .when(col("product_cost") < col("sales_amount"), "cost < sales")
    .otherwise("unknown")
)

print("Sample rows showing all three price fields:")
display(pattern_df.orderBy("quantity", "order_num").limit(50))

print("\nPattern distribution:")
pattern_df.groupBy("quantity", "cost_vs_price", "cost_vs_sales").count().orderBy("quantity", "count").show(20, truncate=False)

---
## Conclusion: Hypothesis Verification Results

### Join Success ✅

Using substring matching (`prd_key CONTAINS sls_prd_key`), we successfully joined:
* **60,398 sales records** × **397 product records** → **102,328 joined records**

### Hypothesis Test Results

**Original Hypothesis**: When `quantity > 1`, either:
* `sls_price = sls_sales / quantity`, OR
* `sls_sales = sls_price * quantity`

**Result: BOTH hypotheses are FALSE** ❌

### Key Findings

#### 1. For Quantity = 1 (102,311 records)

* **sls_price == sls_sales**: TRUE for 99.9% of records (102,255 / 102,311)
* **prd_cost == sls_price**: FALSE (mismatched)
* **prd_cost == sls_sales**: FALSE (mismatched)
* **Pattern**: `prd_cost < sls_price == sls_sales` (102,259 records)

This confirms that **both `sls_price` and `sls_sales` represent the UNIT PRICE sold to the customer**, NOT the product cost or a calculated total.

#### 2. For Quantity > 1 (17 records)

* **Hypothesis 1** (`sls_sales = sls_price * quantity`): **0 matches** (0/17)
* **Hypothesis 2** (`sls_price = sls_sales / quantity`): **0 matches** (0/17)
* **Pattern**: Where values exist, `sls_price == sls_sales` (same unit price behavior)
* Most records (11/17) have NULL in `sls_price`, with only `sls_sales` populated

#### 3. Price Relationship Summary

```
prd_cost (product cost from supplier/manufacturer)
    ↓
    ↓  markup applied
    ↓
sls_price = sls_sales (unit selling price to customer)
    ×
  quantity
    ↓
  [MISSING: total_amount = unit_price × quantity]
```

### Data Quality Issues Identified

1. **Redundant columns**: `sls_price` and `sls_sales` are duplicates (both contain unit price)
2. **Missing calculation**: No column captures the actual total sales amount (unit_price × quantity)
3. **Null handling**: When `quantity > 1`, nulls appear inconsistently across the duplicate columns
4. **Business logic gap**: The data model lacks a clear distinction between unit price and total amount

### Recommendation

For the silver layer transformation:
* **Keep one unit price column** (choose `sls_price` or `sls_sales`, coalesce to handle nulls)
* **Calculate total sales amount** = `COALESCE(sls_price, sls_sales) * quantity`
* Document that `prd_cost` represents supplier cost, while `sls_price`/`sls_sales` represents customer price